In [6]:
import glob
import pandas as pd
import os
from pathlib import Path
import json
import re


def get_allfiles_metadata_json(ignore_list=None):
    if ignore_list is None:
        ignore_list = []

    path_processing = os.getcwd()

    json_files = glob.glob(path_processing + '/**/*.json', recursive=True)

    # ✅ Only timestamped ptycho files
    pattern = re.compile(r"ptycho_.*_\d{8}-\d{6}\.json$")
    json_files = [f for f in json_files if pattern.search(Path(f).name)]

    if ignore_list:
        for this_string in ignore_list:
            json_files = [
                p for p in json_files
                if this_string not in Path(p).parts
            ]

    return json_list_to_dataframe(json_files)


# -------------------------
# JSON parsing
# -------------------------
def extract_json_key_values(fp):
    with open(fp, "r") as f:
        data = json.load(f)

    return data, flatten_dict(data)


def flatten_dict(d, parent_key='', sep='.'):
    items = []

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k

        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep).items())

        elif isinstance(v, list):
            # numeric list expansion
            for i, val in enumerate(v):
                items.append((f"{new_key}_{i}", val))

        else:
            items.append((new_key, v))

    return dict(items)


# -------------------------
# Extract Session / Sample / dataset
# -------------------------
def extract_structure_from_basedir(base_dir):
    """
    Parse:
    /.../2026/mg44072-2/processing/Merlin/TEB30_A1_old/20260512_151123/...
    """

    parts = Path(base_dir).parts

    session = None
    sample = None
    dataset = None

    # ✅ Session: mg#####-#
    for p in parts:
        if re.match(r"mg\d{5}-\d", p):
            session = p
            break

    # ✅ Sample: after 'Merlin'
    if "Merlin" in parts:
        idx = parts.index("Merlin")
        if idx + 1 < len(parts):
            sample = parts[idx + 1]

        # ✅ dataset: next folder after sample
        if idx + 2 < len(parts):
            dataset = parts[idx + 2]

    return session, sample, dataset


# -------------------------
# Main DataFrame builder
# -------------------------
def json_list_to_dataframe(file_list):
    rows = []

    for fp in file_list:
        fp = Path(fp)
        print(fp)

        raw_json, metadata = extract_json_key_values(fp)

        # ✅ Extract base_dir info
        base_dir = raw_json.get("base_dir", "")
        session, sample, dataset = extract_structure_from_basedir(base_dir)

        row = {
            "Session": session,
            "Sample": sample,
            "data-set": dataset,
            "source_file": fp.name,
            **metadata
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [7]:
df = get_allfiles_metadata_json()

/dls/e02/data/2026/mg44072-2/processing/Merlin/TEB30_A1_old/20260512_151123/pty_out/initial_recon/ptycho_0_0_20260513-105257.json
/dls/e02/data/2026/mg44072-2/processing/Merlin/TEB30_A1_old/20260512_151123/pty_out/initial_recon/ptycho_0_0_20260513-100658.json
/dls/e02/data/2026/mg44072-2/processing/Merlin/TEB30_A1_old/20260512_151123/pty_out/initial_recon/ptycho_0_0_20260513-091634.json
/dls/e02/data/2026/mg44072-2/processing/Merlin/TEB30_A1_old/20260512_151123/pty_out/initial_recon/ptycho_0_0_20260513-140825.json
/dls/e02/data/2026/mg44072-2/processing/Merlin/TEB30_A1_old/20260513_114052/pty_out/initial_recon/ptycho_0_0_20260513-123938.json
/dls/e02/data/2026/mg44072-2/processing/Merlin/TEB30_A1_old/20260513_114052/pty_out/initial_recon/ptycho_0_0_20260513-143658.json
/dls/e02/data/2026/mg44072-2/processing/Merlin/TEB30_A1_old/20260512_150717/pty_out/initial_recon/ptycho_0_0_20260513-105257.json
/dls/e02/data/2026/mg44072-2/processing/Merlin/TEB30_A1_old/20260512_150717/pty_out/initia

In [8]:
df

,Session,Sample,data-set,source_file,base_dir,experiment.data.dask,experiment.data.data_key,experiment.data.data_path,experiment.data.dead_pixel_flag,experiment.data.dead_pixel_path,...,process.common.source.energy_0,process.common.source.flux,process.common.source.radiation,process.cores,process.gpu_flag,process.interaction,process.save_dir,process.save_interval,process.save_prefix,process.subset_type
0,mg44072-2,TEB30_A1_old,20260512_151123,ptycho_0_0_20260513-105257.json,/dls/e02/data/2026/mg44072-2/processing/Merlin...,0,dls,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5,...,80000.0,-1,electron,4,1,matplotlib,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,ptycho,random
1,mg44072-2,TEB30_A1_old,20260512_151123,ptycho_0_0_20260513-100658.json,/dls/e02/data/2026/mg44072-2/processing/Merlin...,0,dls,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5,...,80000.0,-1,electron,4,1,matplotlib,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,ptycho,random
2,mg44072-2,TEB30_A1_old,20260512_151123,ptycho_0_0_20260513-091634.json,/dls/e02/data/2026/mg44072-2/processing/Merlin...,0,dls,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5,...,80000.0,-1,electron,4,1,matplotlib,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,ptycho,random
3,mg44072-2,TEB30_A1_old,20260512_151123,ptycho_0_0_20260513-140825.json,/dls/e02/data/2026/mg44072-2/processing/Merlin...,0,dls,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5,...,80000.0,-1,electron,4,1,matplotlib,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,ptycho,random
4,mg44072-2,TEB30_A1_old,20260513_114052,ptycho_0_0_20260513-123938.json,/dls/e02/data/2026/mg44072-2/processing/Merlin...,0,dls,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5,...,80000.0,-1,electron,4,1,matplotlib,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,ptycho,random
5,mg44072-2,TEB30_A1_old,20260513_114052,ptycho_0_0_20260513-143658.json,/dls/e02/data/2026/mg44072-2/processing/Merlin...,0,dls,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5,...,80000.0,-1,electron,4,1,matplotlib,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,ptycho,random
6,mg44072-2,TEB30_A1_old,20260512_150717,ptycho_0_0_20260513-105257.json,/dls/e02/data/2026/mg44072-2/processing/Merlin...,0,dls,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5,...,80000.0,-1,electron,4,1,matplotlib,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,ptycho,random
7,mg44072-2,TEB30_A1_old,20260512_150717,ptycho_0_0_20260513-091631.json,/dls/e02/data/2026/mg44072-2/processing/Merlin...,0,dls,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5,...,80000.0,-1,electron,4,1,matplotlib,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,ptycho,random
8,mg44072-2,TEB30_A1_old,20260512_150717,ptycho_0_0_20260513-100659.json,/dls/e02/data/2026/mg44072-2/processing/Merlin...,0,dls,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5,...,80000.0,-1,electron,4,1,matplotlib,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,ptycho,random
9,mg44072-2,TEB30_A1_old,20260512_150717,ptycho_0_0_20260513-140829.json,/dls/e02/data/2026/mg44072-2/processing/Merlin...,0,dls,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5,...,80000.0,-1,electron,4,1,matplotlib,/dls/e02/data/2026/mg44072-2/processing/Merlin...,1,ptycho,random


In [9]:
#and save to excell
df.to_excel("ptycho_output.xlsx", )